In [ ]:
# =============================================================================
#  片段 1: 环境设置与库导入 (无变化)
# =============================================================================
# 确保已安装最新版本的库
!pip install earthengine-api pandas requests --upgrade

import ee
import pandas as pd
import requests
from datetime import datetime
import time
import io

print("✅ 片段 1: 库已成功安装并导入。")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.6/467.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: earthengine-api
    Found existing installation: earthengine-api 1.5.24
    Uninstalling earthengine-api-1.5.24:
      Successfully uninstalled earthengine-api-1.5.24
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
go

In [ ]:
# =============================================================================
#  片段 2: GEE授权与初始化 (无变化)
# =============================================================================
try:
    # 尝试使用已有的凭证和项目进行初始化
    # 请将 'YOUR_GEE_PROJECT' 替换为您自己的GEE项目ID
    ee.Initialize(project='awaran-20130924')
    print("✅ GEE 初始化成功！")
except Exception as e:
    # 如果初始化失败，则提示进行授权
    print("GEE 需要授权...")
    ee.Authenticate()
    ee.Initialize(project='awaran-20130924')
    print("✅ GEE 授权并初始化成功！")

GEE 需要授权...
✅ GEE 授权并初始化成功！


In [ ]:
# =============================================================================
#  片段 3: 获取并筛选飓风数据 (【最终修正版 v6 - 正确的读取逻辑】)
# =============================================================================
import pandas as pd
from datetime import datetime
import requests
import io

def fetch_and_filter_hurricanes(start_year=2012, min_category=3):
    """
    从NOAA IBTrACS数据库获取并筛选全球热带气旋数据。
    筛选标准: 3级及以上飓风 (萨菲尔-辛普森等级)
    """
    print("正在从NOAA IBTrACS数据库下载全球热带气旋数据 (这可能需要1-2分钟)...")
    url = "https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r00/access/csv/ibtracs.ALL.list.v04r00.csv"

    try:
        response = requests.get(url, timeout=180)
        response.raise_for_status()

        # =======================【核心修正点】=======================
        # 根据您的环境返回的数据，文件从第一行开始就是列名。
        # 因此，我们不跳过任何行，直接读取文本。
        # 同时，我们强制指定分隔符为逗号，以解决最初的解析问题。
        csv_data = pd.read_csv(
            io.StringIO(response.text),
            sep=',',
            na_values=[' '],
            low_memory=False  # 使用默认C引擎时，设置low_memory=False以消除DtypeWarning
        )
        # ==========================================================

        print("数据下载完成，开始处理...")

        csv_data.columns = csv_data.columns.str.strip()

        # --- 数据清洗和类型转换 ---
        csv_data.rename(columns={'SID': 'sid', 'NAME': 'name', 'ISO_TIME': 'time',
                                 'LAT': 'lat', 'LON': 'lon', 'USA_SSHS': 'category'}, inplace=True)

        required_cols = ['sid', 'name', 'time', 'lat', 'lon', 'category']
        if not all(col in csv_data.columns for col in required_cols):
            missing_cols = [col for col in required_cols if col not in csv_data.columns]
            # 增加更详细的调试信息
            print("🔴 错误：重命名前的列名: ", [col for col in csv_data.columns if 'USA_' in col or 'ISO_' in col or 'SID' in col])
            raise ValueError(f"错误：源数据中缺少关键列或重命名失败。缺失列: {missing_cols}")

        csv_data['time'] = pd.to_datetime(csv_data['time'])
        for col in ['lat', 'lon', 'category']:
            csv_data[col] = pd.to_numeric(csv_data[col], errors='coerce')

        csv_data.dropna(subset=['time', 'lat', 'lon', 'category'], inplace=True)

        # --- 按时间和强度筛选 ---
        df_filtered = csv_data[csv_data['time'].dt.year >= start_year].copy()
        max_category_per_storm = df_filtered.groupby('sid')['category'].max()
        major_hurricane_sids = max_category_per_storm[max_category_per_storm >= min_category].index
        df_major_hurricanes = df_filtered[df_filtered['sid'].isin(major_hurricane_sids)].copy()

        # --- 组织数据结构 ---
        hurricanes = []
        for sid, group in df_major_hurricanes.groupby('sid'):
            group = group.sort_values('time')
            start_date = group['time'].min().strftime('%Y-%m-%d')
            max_cat = int(group['category'].max())
            name = group['name'].iloc[0]
            track = [[row['lon'], row['lat']] for index, row in group.iterrows()]

            hurricanes.append({
                'sid': sid,
                'name': name,
                'start_date': start_date,
                'max_category': max_cat,
                'track_line': track
            })

        print(f"✅ 成功筛选出 {len(hurricanes)} 场 {start_year} 年以来, 达到或超过 {min_category} 级的热带气旋。")
        return hurricanes

    except requests.exceptions.RequestException as e:
        print(f"🔴 错误: 无法从NOAA获取飓风数据: {e}")
        return []
    except (ValueError, KeyError) as e:
        print(f"🔴 错误: 处理数据时发生错误: {e}")
        if 'csv_data' in locals():
            print("读取到的列名如下: ", csv_data.columns.tolist())
        return []

# 执行函数
raw_hurricanes = fetch_and_filter_hurricanes(start_year=2012, min_category=3)

正在从NOAA IBTrACS数据库下载全球热带气旋数据 (这可能需要1-2分钟)...
数据下载完成，开始处理...
✅ 成功筛选出 316 场 2012 年以来, 达到或超过 3 级的热带气旋。


In [ ]:
# =============================================================================
#  片段 3.5: 云端预筛选 (【已修改】按飓风路径影响人口)
# =============================================================================
def pre_filter_hurricanes_for_urban_impact(hurricanes, pop_threshold=500000, buffer_radius=300000): # <-- 【核心修改点】
    """
    在GEE服务器端，根据飓风路径影响廊道内的人口数量，预先筛选掉影响较小的事件。
    """
    if not hurricanes:
        return []

    print(f"\n--- 开始在GEE上进行城市影响预筛选 (人口阈值: {pop_threshold/1e3:.0f} 千) ---")
    print(f"    (每个飓风路径将被缓冲 {buffer_radius/1e3:.0f} 公里以估算影响范围)")

    # 将本地的飓风数据转换为GEE特征集合
    features = []
    for h in hurricanes:
        # GEE需要经度在前，纬度在后
        line = ee.Geometry.LineString(h['track_line'])
        # 将元数据作为属性存储
        feature = ee.Feature(line, {k: v for k, v in h.items() if k != 'track_line'})
        features.append(feature)
    h_collection = ee.FeatureCollection(features)

    population_image = ee.ImageCollection("WorldPop/GP/100m/pop").filter(ee.Filter.date('2020-01-01', '2020-12-31')).mean()

    def calculate_population(feature):
        # 对飓风的线状路径进行缓冲，形成一个面状的影响区域
        aoi = feature.geometry().buffer(buffer_radius)
        pop_sum_dict = population_image.reduceRegion(
            reducer=ee.Reducer.sum(), geometry=aoi, scale=1000, maxPixels=1e9
        )
        return feature.set('population_in_aoi', pop_sum_dict.get('population'))

    pop_collection = h_collection.map(calculate_population)
    filtered_collection = pop_collection.filter(ee.Filter.And(
        ee.Filter.notNull(['population_in_aoi']),
        ee.Filter.gt('population_in_aoi', pop_threshold)
    ))

    print("  -> 正在从GEE获取筛选结果...")
    # 安全地获取结果
    size = filtered_collection.size()
    list_of_features = ee.List(ee.Algorithms.If(
        size.gt(0), filtered_collection.toList(size), ee.List([])
    ))

    # 将GEE结果转回Python字典列表
    filtered_list_of_dicts = list_of_features.getInfo()
    final_hurricanes = [f['properties'] for f in filtered_list_of_dicts]
    # 将GEE返回的线状几何体重新加回到字典中，以备后用
    for i, f in enumerate(filtered_list_of_dicts):
        final_hurricanes[i]['track_line'] = f['geometry']['coordinates']


    if not final_hurricanes:
        print("✅ 预筛选完成！在您的数据集中未发现对大城市区域有潜在影响的强飓风。")
    else:
        print(f"✅ 预筛选完成！从 {len(hurricanes)} 场事件中筛选出 {len(final_hurricanes)} 场对城市/城镇有潜在影响的强飓风。")

    return final_hurricanes

# 执行预筛选
if 'raw_hurricanes' in locals():
    hurricanes_to_process = pre_filter_hurricanes_for_urban_impact(raw_hurricanes, pop_threshold=500000, buffer_radius=300000)
    if hurricanes_to_process:
        print("\n有效飓风数据预览：")
        # 为了更好地展示，我们不显示冗长的 track_line
        preview_df = pd.DataFrame(hurricanes_to_process)
        display(preview_df.drop(columns=['track_line']).head())


--- 开始在GEE上进行城市影响预筛选 (人口阈值: 500 千) ---
    (每个飓风路径将被缓冲 300 公里以估算影响范围)
  -> 正在从GEE获取筛选结果...
✅ 预筛选完成！从 316 场事件中筛选出 100 场对城市/城镇有潜在影响的强飓风。

有效飓风数据预览：


,max_category,name,population_in_aoi,sid,start_date
0,4,GUCHOL,9.640613e+05,2012162N06150,2012-06-10
1,4,VICENTE,1.866895e+06,2012201N15129,2012-07-18
2,4,TEMBIN,1.309354e+06,2012230N21126,2012-08-17
3,4,BOLAVEN,1.430256e+06,2012232N13141,2012-08-18
4,5,SANBA,7.957355e+05,2012254N09135,2012-09-10


In [ ]:
# =============================================================================
#  片段 4: 核心GEE分析函数 (按周合成) (【已修改】以处理线状路径)
# =============================================================================
from datetime import datetime

def process_one_hurricane_on_server(feature):
    # 【修改】SEARCH_RADIUS现在用于定义路径周围的分析区域
    SEARCH_RADIUS = 300000  # 300 km
    NTL_CAP = 1000
    NTL_BAND_NAME = 'DNB_BRDF_Corrected_NTL'

    def mask_vnp_quality(image):
        qf_band = image.select('QF_Cloud_Mask')
        clear_mask = qf_band.bitwiseAnd(1 << 0).eq(0).And(qf_band.bitwiseAnd(1 << 1).eq(0))
        return image.updateMask(clear_mask).select(NTL_BAND_NAME)

    # 【修改】这里的geometry现在是飓风的路径（LineString）
    search_area = feature.geometry().buffer(SEARCH_RADIUS)

    # 在这个大的影响范围内，找到所有的城市区域
    urban_polygons = ee.Image('JRC/GHSL/P2023A/GHS_SMOD/2020').select('smod_code') \
                       .gte(2).selfMask().reduceToVectors(geometry=search_area, scale=500, maxPixels=1e10)

    # 只有当影响范围内存在城市时，才继续分析
    def perform_analysis():
        # 将所有碎片化的城市斑块合并成一个大的分析区域
        urban_aoi = urban_polygons.union(100)
        start_date = ee.Date('2012-01-01')
        end_date = ee.Date('2025-01-01')
        n_weeks = end_date.difference(start_date, 'week').ceil()
        weeks = ee.List.sequence(0, n_weeks.subtract(1))

        def get_weekly_sum(week_offset):
            current_start = start_date.advance(week_offset, 'week')
            current_end = current_start.advance(1, 'week')
            # 使用 search_area (路径缓冲带) 来筛选影像，效率更高
            weekly_image = ee.ImageCollection('NASA/VIIRS/002/VNP46A2') \
                              .filterBounds(search_area) \
                              .filterDate(current_start, current_end) \
                              .map(mask_vnp_quality) \
                              .median() # 使用中值合成以去除异常值
            def compute_sum(img):
                # 【修改】仅在 urban_aoi (城市区域) 内进行加总
                sum_dict = ee.Image(img).lte(NTL_CAP).reduceRegion(
                    reducer=ee.Reducer.sum(), geometry=urban_aoi, scale=500, maxPixels=1e13
                )
                return sum_dict.get(NTL_BAND_NAME, -999) # 如果无数据，返回-999

            # 检查合成后的影像是否有波段，避免对空影像操作
            sum_value = ee.Algorithms.If(weekly_image.bandNames().size().gt(0), compute_sum(weekly_image), -999)
            prop_name = ee.String('NTL_').cat(current_start.format('YYYY_ww'))
            return prop_name, sum_value

        time_series_list = weeks.map(get_weekly_sum)
        return feature.set(ee.Dictionary(time_series_list.flatten()))

    # 如果在影响区内没有找到城市，则返回一个空特征，后续会被过滤掉
    return ee.Feature(ee.Algorithms.If(urban_polygons.size().gt(0), perform_analysis(), None), feature.toDictionary())

print("✅ 片段 4: GEE核心分析函数已准备就绪 (适用于飓风路径)。")

✅ 片段 4: GEE核心分析函数已准备就绪 (适用于飓风路径)。


In [ ]:
# =============================================================================
#  片段 5: 服务器端批处理工作流与导出 (【已修改】变量与文件名)
# =============================================================================
import time

def server_side_batch_export_workflow(hurricanes, batch_size=5):
    if not hurricanes:
        print("\n🔴 经过预筛选，没有发现对城市/城镇有显著影响的飓风，任务终止。")
        return

    num_hurricanes = len(hurricanes)
    print(f"\n--- 将对 {num_hurricanes} 场有效飓风进行精确夜光分析 ---")
    total_batches = (num_hurricanes + batch_size - 1) // batch_size

    for i in range(0, num_hurricanes, batch_size):
        batch_num = (i // batch_size) + 1
        current_batch_data = hurricanes[i:i + batch_size]
        print(f"\n--- 正在准备批次 {batch_num} / {total_batches} (包含 {len(current_batch_data)} 个事件) ---")

        features = []
        for h in current_batch_data:
            line = ee.Geometry.LineString(h['track_line'])
            # 移除track_line，因为它过于庞大，不适合作为属性直接导出
            properties = {k: v for k, v in h.items() if k != 'track_line'}
            features.append(ee.Feature(line, properties))

        h_collection = ee.FeatureCollection(features)

        print("  -> 正在将计算任务发送到GEE服务器...")
        processed_collection = h_collection.map(process_one_hurricane_on_server, opt_dropNulls=True)

        # 【文件名更新】
        base_filename = 'urban_hurricane_NTL_weekly_Cat3plus_pop500k'
        output_filename = f"{base_filename}_batch_{batch_num}_of_{total_batches}"

        task = ee.batch.Export.table.toDrive(
            collection=processed_collection,
            description=output_filename,
            folder='GEE_Hurricane_Exports',  # 建议使用新文件夹
            fileNamePrefix=output_filename,
            fileFormat='CSV'
        )
        task.start()
        print(f"🎉 任务 '{output_filename}' 已成功提交！")
        if total_batches > 1:
            time.sleep(5)  # 在提交多个任务时稍作停顿

    print(f"\n\n✅ 所有 {total_batches} 个批处理任务均已提交。请前往您的 Google Drive 'GEE_Hurricane_Exports' 文件夹查看结果。")

if 'hurricanes_to_process' in locals() and hurricanes_to_process:
    server_side_batch_export_workflow(hurricanes_to_process, batch_size=5)
else:
    print("\n🔴 未找到可处理的飓风。请检查片段3和3.5的筛选逻辑。")


--- 将对 100 场有效飓风进行精确夜光分析 ---

--- 正在准备批次 1 / 20 (包含 5 个事件) ---
  -> 正在将计算任务发送到GEE服务器...
🎉 任务 'urban_hurricane_NTL_weekly_Cat3plus_pop500k_batch_1_of_20' 已成功提交！

--- 正在准备批次 2 / 20 (包含 5 个事件) ---
  -> 正在将计算任务发送到GEE服务器...
🎉 任务 'urban_hurricane_NTL_weekly_Cat3plus_pop500k_batch_2_of_20' 已成功提交！

--- 正在准备批次 3 / 20 (包含 5 个事件) ---
  -> 正在将计算任务发送到GEE服务器...
🎉 任务 'urban_hurricane_NTL_weekly_Cat3plus_pop500k_batch_3_of_20' 已成功提交！

--- 正在准备批次 4 / 20 (包含 5 个事件) ---
  -> 正在将计算任务发送到GEE服务器...
🎉 任务 'urban_hurricane_NTL_weekly_Cat3plus_pop500k_batch_4_of_20' 已成功提交！

--- 正在准备批次 5 / 20 (包含 5 个事件) ---
  -> 正在将计算任务发送到GEE服务器...
🎉 任务 'urban_hurricane_NTL_weekly_Cat3plus_pop500k_batch_5_of_20' 已成功提交！

--- 正在准备批次 6 / 20 (包含 5 个事件) ---
  -> 正在将计算任务发送到GEE服务器...
🎉 任务 'urban_hurricane_NTL_weekly_Cat3plus_pop500k_batch_6_of_20' 已成功提交！

--- 正在准备批次 7 / 20 (包含 5 个事件) ---
  -> 正在将计算任务发送到GEE服务器...
🎉 任务 'urban_hurricane_NTL_weekly_Cat3plus_pop500k_batch_7_of_20' 已成功提交！

--- 正在准备批次 8 / 20 (包含 5 个事件) ---
  -> 正在将计算任务发送到GEE服